In [2]:
import pandas as pd
import geopandas as gpd
import altair as alt
from pathlib import Path

In [159]:
MERGED_SF_TRACTS_SHP = (
    Path.cwd().parent / "clean-data/merged_sf_shapefiles/merged_sf_tracts.shp"
)

MERGED = Path.cwd().parent / "clean-data/merged_data.csv"

In [154]:
def create_tract_map(start_date: str, end_date: str, col_name: str):
    """
    Add docstring
    """
    METRIC_NAMES = {
        "311_calls": "311 calls",
        "eviction_rate": "Eviction rate",
        "median_rent": "Median rent",
        "tents": "Tents",
        "structures": "Structures",
        "vehicles": "Vehicles",
        "estimate": "Homelessness estimate",
    }

    merged_df = pd.read_csv(MERGED)
    tracts_gdf = gpd.read_file(MERGED_SF_TRACTS_SHP)

    merged_df["tract"] = merged_df["tract"].astype(str).str.zfill(11)

    # Filter
    filtered_df = (
        merged_df[(merged_df["date"] >= start_date) & (merged_df["date"] <= end_date)]
        .drop(columns=["date"])
        .groupby("tract")
        .mean()
        .reset_index()
    )

    # Merge filtered, aggregated dataframe with GeoDataFrame
    merged_tracts_gdf = tracts_gdf.merge(
        filtered_df, left_on="GEOID", right_on="tract", how="left"
    )

    tooltips = [
        alt.Tooltip("GEOID:N", title="Tract ID"),
        alt.Tooltip("population:Q", title="Population"),
        alt.Tooltip("median_rent:Q", format=",.0f", title="Median rent (per month)"),
        alt.Tooltip("eviction_rate:Q", format=".3%", title="Average eviction rate"),
        alt.Tooltip(
            "estimate:Q", format=",.0f", title="Average homeless population estimate"
        ),
        alt.Tooltip(
            "311_calls:Q",
            format=",.0f",
            title="Average monthly citizen-reported encampments",
        ),
    ]

    if col_name in ["tents", "structures", "vehicles"]:
        tooltips.append(
            alt.Tooltip(
                f"{col_name}:Q",
                format=",.0f",
                title=f"Average {METRIC_NAMES[col_name].lower()}",
            )
        )

    # Create base map for tracts with no data
    background = (
        alt.Chart(tracts_gdf)
        .mark_geoshape(fill="lightgray", stroke="white")
        .project("albersUsa")
    )

    # Build chart
    chart = (
        alt.Chart(merged_tracts_gdf)
        .mark_geoshape()
        .encode(
            color=alt.Color(f"{col_name}:Q", title={METRIC_NAMES[col_name]}),
            tooltip=tooltips,
        )
        .transform_lookup(
            lookup="GEOID",
            from_=alt.LookupData(filtered_df, ("tract"), ["metric"]),
        )
        .project(type="albersUsa")
        .properties(
            title=f"Average {METRIC_NAMES[col_name].lower()} in SF tracts ({start_date} to {end_date})"
        )
        .interactive()
    )

    return background + chart

In [ ]:
def create_tract_map(start_date: str, end_date: str, col_name: str):
    """
    Create Altair choropleth map of SF tracts based on a specified metric
    averaged given a start date and end date.

    Parameters:
        start_date: Start date for analysis
        end_date: End date for analysis
        col_name: Column name for preferred metric of analysis

    Returns:
        An Altair choropleth map across SF tracts for the given metric and time
        period for analysis
    """
    METRIC_NAMES = {
        "311_calls": "311 calls",
        "eviction_rate": "Eviction rate",
        "median_rent": "Median rent",
        "tents": "Tents",
        "structures": "Structures",
        "vehicles": "Vehicles",
        "estimate": "Homelessness estimate",
    }

    merged_df = pd.read_csv(MERGED)
    tracts_gdf = gpd.read_file(MERGED_SF_TRACTS_SHP)

    merged_df["tract"] = merged_df["tract"].astype(str).str.zfill(11)

    filtered_df = (
        merged_df[(merged_df["date"] >= start_date) & (merged_df["date"] <= end_date)]
        .drop(columns=["date"])
        .groupby("tract")
        .mean()
        .reset_index()
    )

    # Merge aggregated dataframe with GeoDataFrame
    merged_tracts_gdf = tracts_gdf.merge(
        filtered_df, left_on="GEOID", right_on="tract", how="left"
    )

    # Create tooltips, and add last tooltip only if selected column name is not one of the existing tooltips
    tooltips = [
        alt.Tooltip("GEOID:N", title="Tract ID"),
        alt.Tooltip("population:Q", title="Population"),
        alt.Tooltip("median_rent:Q", format=",.0f", title="Median rent (per month)"),
        alt.Tooltip("eviction_rate:Q", format=".3%", title="Average eviction rate"),
        alt.Tooltip(
            "estimate:Q", format=",.0f", title="Average homeless population estimate"
        ),
        alt.Tooltip(
            "311_calls:Q",
            format=",.0f",
            title="Average monthly citizen-reported encampments",
        ),
    ]

    if col_name in ["tents", "structures", "vehicles"]:
        tooltips.append(
            alt.Tooltip(
                f"{col_name}:Q",
                format=",.0f",
                title=f"Average {METRIC_NAMES[col_name].lower()}",
            )
        )

    # Create base map for tracts with no data
    background = (
        alt.Chart(tracts_gdf)
        .mark_geoshape(fill="lightgray", stroke="white")
        .project("albersUsa")
    )

    # Build interactive choropleth map
    chart = (
        alt.Chart(merged_tracts_gdf)
        .mark_geoshape(stroke="black", strokeWidth=0.5)
        .encode(
            color=alt.Color(
                f"{col_name}:Q",
                title=METRIC_NAMES[col_name],
                legend=alt.Legend(
                    orient="right",
                    direction="vertical",
                    labelAlign="left",
                    titlePadding=5,
                    offset=10,
                ),
            ),
            tooltip=tooltips,
        )
        .transform_lookup(
            lookup="GEOID",
            from_=alt.LookupData(filtered_df, "tract", ["metric"]),
        )
        .project("albersUsa")
        # .properties(width="container", height=550)
        # .configure_view(stroke=None)
        .interactive()
    )

    return background + chart

In [203]:
create_tract_map("2020-01", "2024-12", "tents")

TypeError: Objects with 'config' attribute cannot be used within LayerChart. Consider defining the config attribute in the LayerChart object instead.

In [178]:
def create_homeless_scatterplot(tract_id: str):
    """
    Add docstring
    """

    df = pd.read_csv(MERGED)

    df["tract"] = df["tract"].astype(str).str.zfill(11)

    filtered_df = df[df["tract"] == tract_id]

    chart = (
        alt.Chart(filtered_df)
        .mark_line(point=True)
        .encode(
            x=alt.X("date:T", title="Date"),
            y=alt.Y("estimate:Q", title="Homeless Population Estimate"),
        )
        .properties(
            title=f"Homeless population estimate over time in census tract {tract_id}"
        )
    )

    return chart

In [179]:
create_homeless_scatterplot("06075980900")

alt.Chart(...)

In [180]:
CLEAN_ZILLOW = Path.cwd().parent / "clean-data/clean_zillow_data.csv"


def create_rent_scatterplot(zip_code: str):
    df = pd.read_csv(CLEAN_ZILLOW)

    df["zip"] = df["zip"].astype(str).str.zfill(5)

    filtered_df = df[df["zip"] == zip_code]

    chart = (
        alt.Chart(filtered_df)
        .mark_line(point=True)
        .encode(
            x=alt.X("date:T", title="Date"),
            y=alt.Y("rent:Q", title="Median rent (per month)"),
        )
        .properties(title=f"Median rent (per month) over time in zip code {zip_code}")
    )

    return chart

In [181]:
create_rent_scatterplot("94102")

alt.Chart(...)

In [184]:
def create_encampments_scatterplot(tract_id: str):
    df = pd.read_csv(MERGED)

    df["tract"] = df["tract"].astype(str).str.zfill(11)

    filtered_df = df[df["tract"] == tract_id]

    folded_chart = (
        alt.Chart(filtered_df)
        .mark_line()
        .transform_fold(
            fold=["structures", "tents", "vehicles"],
            as_=["measurement", "value"],
        )
        .encode(
            x=alt.X("date:T"),
            y=alt.Y("value:Q"),
            color=alt.Color("measurement:N"),
        )
    )

    return folded_chart

In [185]:
create_encampments_scatterplot("06075980900")

alt.Chart(...)